In [2]:
# req
import pandas as pd
import re
import numpy as np
import os

In [3]:
# a. Patroonherkenning en parsing
MONTH_NAMES = {
    'januari': 1, 'jan': 1, 'january': 1,
    'februari': 2, 'feb': 2, 'february': 2,
    'maart': 3, 'mrt': 3, 'march': 3, 'mar': 3,
    'april': 4, 'apr': 4,
    'mei': 5, 'may': 5,
    'juni': 6, 'jun': 6, 'june': 6,
    'juli': 7, 'jul': 7, 'july': 7,
    'augustus': 8, 'aug': 8, 'august': 8, 'oogst': 8,
    'september': 9, 'sep': 9, 'sept': 9, '7ber': 9, '7br': 9,
    'oktober': 10, 'oct': 10, 'okt': 10, 'october': 10, '8ber': 10, '8br': 10,
    'november': 11, 'nov': 11, '9ber': 11, '9br': 11,
    'december': 12, 'dec': 12, 'xber': 12, 'xbr': 12, '10ber': 12, '10br': 12,
    'janvier': 1, 'février': 2, 'fevrier': 2, 'mars': 3,
    'avril': 4, 'mai': 5, 'juin': 6, 'juillet': 7,
    'août': 8, 'aout': 8, 'septembre': 9,
    'octobre': 10, 'novembre': 11, 'décembre': 12, 'decembre': 12
}

SPECIAL_WORDS = {
    'dezer': 'current_month', 
    'dezen': 'current_month',
    'dezes': 'current_month',
    'of': 'or',
    'den': 'ordinal',
    'der': 'ordinal',
    'bon': 'ordinal'
}

# c. Validation constants
DEFAULT_MIN_YEAR = 1875
DEFAULT_MAX_YEAR = 1945

# c. Validatie, reconstructie en correctie
def is_valid_date_part(day=None, month=None, year=None, start_year=None, end_year=None):
    result = {'day': True, 'month': True, 'year': True}

    if month is not None:
        if not isinstance(month, int) or month < 1 or month > 12:
            result['month'] = False

    if year is not None:
        min_year = start_year if start_year is not None and not pd.isna(start_year) else DEFAULT_MIN_YEAR
        max_year = end_year if end_year is not None and not pd.isna(end_year) else DEFAULT_MAX_YEAR
        if not isinstance(year, int) or year < min_year or year > max_year:
            result['year'] = False

    if day is not None and month is not None and result['month']:
        days_in_month = [0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        if month == 2 and year is not None and isinstance(year, int) and result['year']:
            if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
                days_in_month[2] = 29
        if not isinstance(day, int) or day < 1 or day > days_in_month[month]:
            result['day'] = False

    return result

def merge_with_previous_date(parsed_date, previous_day=None, previous_month=None, previous_year=None, start_year=None, end_year=None):
    if parsed_date is None:
        return None

    merged_date = parsed_date.copy()

    validity = is_valid_date_part(
        day=parsed_date.get('day'),
        month=parsed_date.get('month'),
        year=parsed_date.get('year'),
        start_year=start_year,
        end_year=end_year
    )

    # als dag ongeldig is, gebruik vorige dag
    if not validity['day'] and previous_day is not None:
        merged_date['day'] = previous_day
    
    # als maand ongeldig is, gebruik vorige maand
    if not validity['month'] and previous_month is not None:
        merged_date['month'] = previous_month

    # als jaar ongeldig is, gebruik vorig jaar
    if not validity['year'] and previous_year is not None:
        merged_date['year'] = previous_year
    elif not validity['year'] and start_year is not None and not pd.isna(start_year):
        merged_date['year'] = int(start_year)
    
    # finale check
    final_validity = is_valid_date_part(
        day=merged_date.get('day'),
        month=merged_date.get('month'),
        year=merged_date.get('year'),
        start_year=start_year,
        end_year=end_year
    )
    
    # als dag nog steeds ongeldig is, gebruik eerste dag van de maand
    if not final_validity['day']:
        merged_date['day'] = 1
        
    # als maand nog steeds ongeldig is, gebruik 1 (januari)
    if not final_validity['month']:
        merged_date['month'] = 1
        
    # als jaar nog steeds ongeldig en we hebben start_year, gebruik start_year
    if not final_validity['year'] and start_year is not None and not pd.isna(start_year):
        merged_date['year'] = int(start_year)
    # als jaar nog steeds ongeldig en we hebben geen start_year, gebruik DEFAULT_MIN_YEAR
    elif not final_validity['year']:
        merged_date['year'] = DEFAULT_MIN_YEAR

    # Final validation to ensure the date is valid
    final_validity = is_valid_date_part(
        day=merged_date.get('day'),
        month=merged_date.get('month'),
        year=merged_date.get('year'),
        start_year=start_year,
        end_year=end_year
    )
    
    return merged_date if all(final_validity.values()) else None

# b. Maandherkenning
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def find_closest_month(month_str, max_distance=2):
    if not isinstance(month_str, str):
        return None
    month_str = month_str.lower().strip()

    min_distance = float('inf')
    closest_month = None
    for month_name in MONTH_NAMES.keys():
        distance = levenshtein_distance(month_str, month_name)
        if distance < min_distance and distance <= max_distance:
            min_distance = distance
            closest_month = month_name
    return MONTH_NAMES.get(closest_month)

def parse_date(date_str, previous_day=None, previous_month=None, previous_year=None, start_year=None, end_year=None):
    if pd.isna(date_str) or not isinstance(date_str, str) or date_str.strip() == '':
        return None

    if ';' in date_str:
        for part in date_str.split(';'):
            parsed = parse_date(part.strip(), previous_day, previous_month, previous_year, start_year, end_year)
            if parsed:
                return parsed
        return None

    date_str = date_str.lower().strip()
    day, month, year = None, None, None

    day_match = re.search(r'\b(\d{1,2})\b', date_str)
    if day_match:
        potential_day = int(day_match.group(1))
        if potential_day > 31:
            if previous_day and 1 <= previous_day <= 31:
                date_str = date_str.replace(day_match.group(0), str(previous_day))
            else:
                date_str = date_str.replace(day_match.group(0), '1')

    # patroonherkenning "dag - maand - jaar"
    dash_pattern = re.search(r'^(\d{1,2})\s*-\s*(\d{1,2})\s*-\s*(\d{1,4})$', date_str)
    if dash_pattern:
        day = int(dash_pattern.group(1))
        month = int(dash_pattern.group(2))
        raw_year = int(dash_pattern.group(3))

        # Handle two-digit years
        if raw_year < 100:
            # If the year is between 0-99, we need to convert it to four digits
            if previous_year is not None:
                # Use previous year's century
                century = (previous_year // 100) * 100
                year = century + raw_year
            elif start_year is not None and not pd.isna(start_year):
                century = int(start_year) // 100 * 100
                year = century + raw_year
            else:
                # Default assumption: 00-99 → 1800-1899
                year = 1800 + raw_year
        elif raw_year < DEFAULT_MIN_YEAR or raw_year > DEFAULT_MAX_YEAR:
            year = previous_year if previous_year is not None else (int(start_year) if start_year is not None and not pd.isna(start_year) else DEFAULT_MIN_YEAR)
        else:
            year = raw_year

        # Instead of directly returning, use merge_with_previous_date to validate
        return merge_with_previous_date(
            {'day': day, 'month': month, 'year': year},
            previous_day, previous_month, previous_year, start_year, end_year
        )
    
    # maandnaam zoeken in string
    for month_name, month_num in MONTH_NAMES.items():
        if month_name in date_str:
            day_match = re.search(r'\b(\d{1,2})\b', date_str)
            
            # Look for both 4-digit and 2-digit years
            year_match_4digit = re.search(r'\b(\d{4})\b', date_str)
            year_match_2digit = None if year_match_4digit else re.search(r'\b(\d{2})\b', date_str)
            
            day = int(day_match.group(1)) if day_match else previous_day
            if day and day > 31:
                day = previous_day if previous_day and previous_day <= 31 else 1
            
            # Handle year based on what was found
            if year_match_4digit:
                year = int(year_match_4digit.group(1))
            elif year_match_2digit:
                year_digits = int(year_match_2digit.group(1))
                if previous_year is not None:
                    # Use the century from the previous year
                    century = (previous_year // 100) * 100
                    year = century + year_digits
                elif start_year is not None and not pd.isna(start_year):
                    century = int(start_year) // 100 * 100
                    year = century + year_digits
                else:
                    # Default assumption
                    year = 1800 + year_digits
            else:
                year = previous_year
                
            return merge_with_previous_date({'day': day, 'month': month_num, 'year': year}, 
                                           previous_day, previous_month, previous_year, start_year, end_year)
    
    # "dezer" en varianten
    dezer_pattern = re.search(r'(\d{1,2})(?:\s+\w+)?\s+(?:dezer|dezen|dezes)', date_str)
    if dezer_pattern:
        day = int(dezer_pattern.group(1))
        if day > 31:
            day = previous_day if previous_day and previous_day <= 31 else 1
            
        return merge_with_previous_date({'day': day, 'month': previous_month, 'year': previous_year}, 
                                       previous_day, previous_month, previous_year, start_year, end_year)
    
    # "15 op 14 dezer"
    op_dezer_pattern = re.search(r'(\d{1,2})\s+op\s+(\d{1,2})\s+(?:dezer|dezen|dezes)', date_str)
    if op_dezer_pattern:
        day = int(op_dezer_pattern.group(1))
        if day > 31:
            day = previous_day if previous_day and previous_day <= 31 else 1
            
        return merge_with_previous_date({'day': day, 'month': previous_month, 'year': previous_year}, 
                                       previous_day, previous_month, previous_year, start_year, end_year)
    

    # "29 op 30 - 4"
    range_match = re.search(r'^(\d{1,2})\s+op\s+(\d{1,2})\s*-\s*(\d{1,2})(?:\s*-\s*(\d{2,4}))?', date_str)
    if range_match:
        day = int(range_match.group(1))
        if day > 31:
            day = previous_day if previous_day and previous_day <= 31 else 1
            
        month = int(range_match.group(3))
        year = int(range_match.group(4)) if range_match.group(4) else previous_year
        
        # Use merge_with_previous_date for validation
        return merge_with_previous_date(
            {'day': day, 'month': month, 'year': year}, 
            previous_day, previous_month, previous_year, start_year, end_year
        )

    # algemene fallback: gegevens van vorige datum
    numbers = re.findall(r'\b(\d{1,2})\b', date_str)
    if numbers:
        day = int(numbers[0])
        if day > 31:
            day = previous_day if previous_day and previous_day <= 31 else 1
            
        return merge_with_previous_date({'day': day, 'month': previous_month, 'year': previous_year}, previous_day, previous_month, previous_year, start_year, end_year)

    return None

# d. Formatering
def format_date(date):
    if not date or not isinstance(date, dict):
        return None

    day = date.get('day')
    month = date.get('month')
    year = date.get('year')

    if day is None or month is None or year is None:
        return None

    try:
        return f"{int(day):02d}/{int(month):02d}/{int(year)}"
    except (TypeError, ValueError):
        return None


# Toepassing
INPUT_FILE = '../../../data/H4/normalisering/machinaal_test.xlsx'
OUTPUT_FILE = '../../../data/H4/normalisering/machinaal_test_date.xlsx'
DATE_COLUMN_NAME = 'E_DATE'
ARCHIVE_COLUMN_NAME = 'archive_file'
CLEANED_COLUMN_NAME = 'E_DATE_clean'
METADATA_FILE = '../../../data/H4/normalisering/IB-metadata.csv'

def clean_dates():
    try:
        metadata = pd.read_csv(METADATA_FILE)
        metadata['date_start'] = pd.to_datetime(metadata['date_start'], dayfirst=True, errors='coerce')
        metadata['date_end'] = pd.to_datetime(metadata['date_end'], dayfirst=True, errors='coerce')

        metadata['start_year'] = metadata['date_start'].dt.year
        metadata['start_month'] = metadata['date_start'].dt.month
        metadata['start_day'] = metadata['date_start'].dt.day
        metadata['end_year'] = metadata['date_end'].dt.year

        df = pd.read_excel(INPUT_FILE)
        
        temp_df = df.merge(metadata[[ARCHIVE_COLUMN_NAME, 'start_year', 'start_month', 'start_day', 'end_year']],
                      on=ARCHIVE_COLUMN_NAME, how='left')

        archive_last_dates = {}
        cleaned_dates = []
        previous_date = None
        current_archive = None

        for idx, row in temp_df.iterrows():
            raw_date = row[DATE_COLUMN_NAME]
            start_year = row['start_year']
            start_month = row['start_month']
            start_day = row['start_day']
            end_year = row['end_year']
            archive_id = row[ARCHIVE_COLUMN_NAME]

            if current_archive != archive_id:
                previous_date = archive_last_dates.get(archive_id)
                if previous_date is None and not pd.isna(start_year) and not pd.isna(start_month):
                    default_day = int(start_day) if not pd.isna(start_day) else 1
                    default_month = int(start_month)
                    default_year = int(start_year)
                    previous_date = {'day': default_day, 'month': default_month, 'year': default_year}
                    archive_last_dates[archive_id] = previous_date.copy()
                current_archive = archive_id

            prev_day = previous_date['day'] if previous_date else None
            prev_month = previous_date['month'] if previous_date else None
            prev_year = previous_date['year'] if previous_date else None

            parsed = parse_date(raw_date, prev_day, prev_month, prev_year, start_year, end_year)

            date_to_format = None
            
            if parsed and all(parsed.get(k) is not None for k in ['day', 'month', 'year']):
                # Additional validation check to ensure month is <= 12
                if parsed['month'] > 12 and prev_month is not None:
                    parsed['month'] = prev_month
                    
                min_year = start_year if start_year is not None and not pd.isna(start_year) else DEFAULT_MIN_YEAR
                max_year = end_year if end_year is not None and not pd.isna(end_year) else DEFAULT_MAX_YEAR
                
                if 100 <= parsed['year'] < 1000 and prev_year is not None:
                    millennium_prefix = (prev_year // 1000) * 1000
                    potential_year = millennium_prefix + parsed['year']
                    if min_year <= potential_year <= max_year:
                        parsed['year'] = potential_year
                
                if parsed['year'] < min_year or parsed['year'] > max_year:
                    if prev_year is not None:
                        parsed['year'] = prev_year
                    elif not pd.isna(start_year):
                        parsed['year'] = int(start_year)
                    else:
                        parsed['year'] = DEFAULT_MIN_YEAR
                
                date_to_format = parsed
                previous_date = parsed.copy()
                archive_last_dates[archive_id] = previous_date.copy()
            elif previous_date:
                date_to_format = previous_date
            elif not pd.isna(start_year):
                default_day = int(start_day) if not pd.isna(start_day) else 1
                default_month = int(start_month) if not pd.isna(start_month) else 1
                default_year = int(start_year)
                date_to_format = {'day': default_day, 'month': default_month, 'year': default_year}
                previous_date = date_to_format.copy()
                archive_last_dates[archive_id] = previous_date.copy()

            formatted = format_date(date_to_format)
            cleaned_dates.append(formatted)

        df[CLEANED_COLUMN_NAME] = cleaned_dates
        
        empty_dates_count = df[CLEANED_COLUMN_NAME].isna().sum()
        if empty_dates_count > 0:
            print(f"Warning: There are still {empty_dates_count} empty dates after cleaning.")
            print("Sample of records with empty dates:")
            print(temp_df[df[CLEANED_COLUMN_NAME].isna()].head(5)[[DATE_COLUMN_NAME, ARCHIVE_COLUMN_NAME, 'start_year', 'start_month']])
        else:
            print("Success: All dates have been successfully cleaned and filled!")
            
        df.to_excel(OUTPUT_FILE, index=False)

    except Exception as e:
        import traceback
        print(f"Fout tijdens datums opschonen: {e}")
        print(traceback.format_exc())

if __name__ == "__main__":
    clean_dates()

Success: All dates have been successfully cleaned and filled!
